In [ ]:
from __future__ import annotations

from collections import defaultdict
from typing import Generator

import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, PreTrainedTokenizer

from tklearn.kb.base import KnowledgeBase
from tklearn.kb.models import Mention, Triple

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
tokenizer.special_tokens_map

In [ ]:
def tokenize(batch: dict) -> dict:
    data = tokenizer(batch["text"], return_offsets_mapping=True)
    batch.update(data)
    return batch

In [ ]:
batch = {
    "text": [
        "Apple is looking at buying U.K. startup for $1 billion",
    ]
}

In [ ]:
batch = tokenize(batch)

In [ ]:
batch

In [ ]:
for document in batch["input_ids"]:
    print(
        "".join("[{}]".format(tokenizer.decode(token)) for token in document)
    )
    print()

In [ ]:
for offset_mappings, item_text in zip(batch["offset_mapping"], batch["text"]):
    char2token = np.full(len(item_text), -1, dtype=np.int32)
    for token_id, (start, end) in enumerate(offset_mappings):
        print(token_id, start, end, item_text[start:end])
        for char_pos in range(start, end):
            char2token[char_pos] = token_id
    print(list(zip(item_text, char2token)))
    print("---")

In [ ]:
import torch
from transformers import BertModel, BertTokenizer

# 1. Load pre-trained model and tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

# 2. Create inputs
text = "Tim Cook Apple"
inputs = tokenizer(text, return_tensors="pt")

# Standard Input IDs: [101, 3845, 5662, 6207, 102] -> ([CLS], Tim, Cook, Apple, [SEP])
input_ids = inputs["input_ids"]

# 3. Create Custom Position IDs
# In a standard sequence, these would be [0, 1, 2, 3, 4]
# For K-BERT "soft positions", you might want "Apple" (index 3) to share "Cook's" position (index 2)
# Custom IDs: [0, 1, 2, 2, 3]
custom_position_ids = torch.tensor([[0, 1, 2, 2, 3]])

# 4. Forward pass with custom positions
outputs = model(
    input_ids=input_ids,
    attention_mask=inputs["attention_mask"],
    position_ids=custom_position_ids,  # <--- Pass your custom positions here
)

print("Output shape:", outputs.last_hidden_state.shape)